# HTB Pentest Agent - Case File Analysis

This notebook provides tools for analyzing case files from penetration testing engagements.

**Use Cases:**
- Post-engagement analysis and reporting
- Cross-case comparisons
- Custom visualizations
- Data export to various formats

In [ ]:
# Setup - Import required modules
import json
import os
from datetime import datetime
from collections import Counter
from pathlib import Path

# Case files directory
CASE_DIR = Path("cases")

# Optional: For visualizations
try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False
    print("matplotlib not available - install with: pip install matplotlib")

## 1. Load Case Files

In [ ]:
def load_case(case_id: str) -> dict:
    """Load a single case file by ID."""
    filepath = CASE_DIR / f"{case_id}.json"
    if filepath.exists():
        with open(filepath, 'r') as f:
            return json.load(f)
    return None

def list_all_cases() -> list:
    """List all available case files."""
    cases = []
    if CASE_DIR.exists():
        for f in CASE_DIR.glob("*.json"):
            try:
                with open(f, 'r') as fp:
                    data = json.load(fp)
                    cases.append({
                        "case_id": data.get("case_id"),
                        "name": data.get("name"),
                        "status": data.get("status"),
                        "created": data.get("created"),
                        "modified": data.get("modified"),
                        "targets": len(data.get("targets", {})),
                        "findings": len(data.get("findings", [])),
                        "credentials": len(data.get("credentials", []))
                    })
            except:
                pass
    return sorted(cases, key=lambda x: x.get("modified", ""), reverse=True)

# Display available cases
cases = list_all_cases()
print(f"Found {len(cases)} case(s):\n")
for c in cases:
    print(f"  [{c['case_id']}] {c['name']}")
    print(f"      Status: {c['status']} | Targets: {c['targets']} | Findings: {c['findings']} | Creds: {c['credentials']}")
    print()

In [ ]:
# Load a specific case (change case_id as needed)
CASE_ID = cases[0]["case_id"] if cases else None

if CASE_ID:
    case = load_case(CASE_ID)
    print(f"Loaded: {case['name']}")
    print(f"Created: {case['created']}")
    print(f"Targets: {len(case.get('targets', {}))}")
    print(f"Findings: {len(case.get('findings', []))}")
else:
    print("No cases found. Run some scans in the GUI first!")
    case = None

## 2. Target Analysis

In [ ]:
def analyze_targets(case: dict) -> dict:
    """Analyze targets from a case file."""
    targets = case.get("targets", {})
    
    all_ports = []
    all_services = []
    os_guesses = []
    
    for addr, data in targets.items():
        for port in data.get("ports", []):
            if port.get("state") == "open":
                all_ports.append(port["port"])
                if port.get("service"):
                    all_services.append(port["service"])
        if data.get("os_guess"):
            os_guesses.append(data["os_guess"])
    
    return {
        "total_targets": len(targets),
        "total_open_ports": len(all_ports),
        "unique_ports": len(set(all_ports)),
        "port_frequency": Counter(all_ports).most_common(10),
        "service_frequency": Counter(all_services).most_common(10),
        "os_guesses": os_guesses
    }

if case:
    analysis = analyze_targets(case)
    print("=== Target Analysis ===")
    print(f"Total Targets: {analysis['total_targets']}")
    print(f"Total Open Ports: {analysis['total_open_ports']}")
    print(f"Unique Ports: {analysis['unique_ports']}")
    print(f"\nTop Ports:")
    for port, count in analysis['port_frequency']:
        print(f"  {port}: {count}x")
    print(f"\nTop Services:")
    for svc, count in analysis['service_frequency']:
        print(f"  {svc}: {count}x")
    if analysis['os_guesses']:
        print(f"\nOS Detections:")
        for os in analysis['os_guesses']:
            print(f"  - {os}")

## 3. Findings Analysis

In [ ]:
def analyze_findings(case: dict) -> dict:
    """Analyze findings by severity and type."""
    findings = case.get("findings", [])
    
    by_severity = Counter(f.get("severity", "info") for f in findings)
    by_type = Counter(f.get("type", "unknown") for f in findings)
    
    # Calculate risk score (weighted)
    weights = {"critical": 10, "high": 7, "medium": 4, "low": 2, "info": 1}
    risk_score = sum(weights.get(f.get("severity", "info"), 1) for f in findings)
    
    return {
        "total": len(findings),
        "by_severity": dict(by_severity),
        "by_type": dict(by_type),
        "risk_score": risk_score,
        "findings": findings
    }

if case:
    findings_analysis = analyze_findings(case)
    print("=== Findings Analysis ===")
    print(f"Total Findings: {findings_analysis['total']}")
    print(f"Risk Score: {findings_analysis['risk_score']}")
    print(f"\nBy Severity:")
    for sev in ["critical", "high", "medium", "low", "info"]:
        count = findings_analysis['by_severity'].get(sev, 0)
        if count > 0:
            emoji = {"critical": "\U0001F534", "high": "\U0001F7E0", "medium": "\U0001F7E1", "low": "\U0001F535", "info": "\u26AA"}[sev]
            print(f"  {emoji} {sev.upper()}: {count}")
    print(f"\nBy Type:")
    for ftype, count in findings_analysis['by_type'].items():
        print(f"  {ftype}: {count}")

In [ ]:
# List critical and high findings
if case:
    print("=== Critical & High Findings ===")
    for f in findings_analysis['findings']:
        if f.get('severity') in ['critical', 'high']:
            print(f"\n[{f['severity'].upper()}] {f['title']}")
            if f.get('target'):
                print(f"  Target: {f['target']}")
            if f.get('description'):
                print(f"  {f['description'][:200]}..." if len(f.get('description', '')) > 200 else f"  {f['description']}")

## 4. Visualizations

In [ ]:
if PLOTTING_AVAILABLE and case:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Severity distribution
    severities = ["critical", "high", "medium", "low", "info"]
    colors = ["#e74c3c", "#e67e22", "#f1c40f", "#3498db", "#95a5a6"]
    counts = [findings_analysis['by_severity'].get(s, 0) for s in severities]
    
    axes[0].bar(severities, counts, color=colors)
    axes[0].set_title("Findings by Severity")
    axes[0].set_ylabel("Count")
    
    # Top ports
    if analysis['port_frequency']:
        ports, port_counts = zip(*analysis['port_frequency'][:8])
        axes[1].barh([str(p) for p in ports], port_counts, color="#2ecc71")
        axes[1].set_title("Top Open Ports")
        axes[1].set_xlabel("Count")
        axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
elif not PLOTTING_AVAILABLE:
    print("Install matplotlib for visualizations: pip install matplotlib")

## 5. Cross-Case Analysis

In [ ]:
def cross_case_analysis(case_ids: list = None) -> dict:
    """Analyze multiple cases for patterns."""
    if case_ids is None:
        case_ids = [c['case_id'] for c in list_all_cases()]
    
    all_ports = []
    all_services = []
    all_findings = []
    total_targets = 0
    
    for cid in case_ids:
        c = load_case(cid)
        if c:
            total_targets += len(c.get('targets', {}))
            for addr, data in c.get('targets', {}).items():
                for port in data.get('ports', []):
                    if port.get('state') == 'open':
                        all_ports.append(port['port'])
                        if port.get('service'):
                            all_services.append(port['service'])
            all_findings.extend(c.get('findings', []))
    
    return {
        "cases_analyzed": len(case_ids),
        "total_targets": total_targets,
        "total_findings": len(all_findings),
        "common_ports": Counter(all_ports).most_common(15),
        "common_services": Counter(all_services).most_common(10),
        "severity_distribution": Counter(f.get('severity', 'info') for f in all_findings)
    }

# Analyze all cases
if cases:
    cross = cross_case_analysis()
    print("=== Cross-Case Analysis ===")
    print(f"Cases Analyzed: {cross['cases_analyzed']}")
    print(f"Total Targets: {cross['total_targets']}")
    print(f"Total Findings: {cross['total_findings']}")
    print(f"\nMost Common Ports Across All Engagements:")
    for port, count in cross['common_ports']:
        print(f"  {port}: {count}x")
    print(f"\nSeverity Distribution:")
    for sev, count in cross['severity_distribution'].items():
        print(f"  {sev}: {count}")

## 6. Custom Report Generation

In [ ]:
def generate_executive_summary(case: dict) -> str:
    """Generate an executive summary for a case."""
    findings = case.get('findings', [])
    targets = case.get('targets', {})
    creds = case.get('credentials', [])
    
    severity_counts = Counter(f.get('severity', 'info') for f in findings)
    
    summary = f"""# Executive Summary: {case.get('name', 'Untitled')}

**Assessment Date:** {case.get('created', 'N/A')[:10]}
**Report Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Overview

This penetration test assessed **{len(targets)} target(s)** and identified **{len(findings)} finding(s)**.

### Risk Summary

| Severity | Count |
|----------|-------|
| Critical | {severity_counts.get('critical', 0)} |
| High | {severity_counts.get('high', 0)} |
| Medium | {severity_counts.get('medium', 0)} |
| Low | {severity_counts.get('low', 0)} |
| Info | {severity_counts.get('info', 0)} |

### Key Findings

"""
    
    # Add critical/high findings
    critical_high = [f for f in findings if f.get('severity') in ['critical', 'high']]
    if critical_high:
        for f in critical_high[:5]:
            summary += f"- **[{f['severity'].upper()}]** {f['title']}\n"
    else:
        summary += "- No critical or high severity findings identified.\n"
    
    if creds:
        summary += f"\n### Credentials Discovered: {len(creds)}\n"
        summary += "\n*Credential details redacted for executive summary.*\n"
    
    summary += "\n---\n*Full technical details available in the complete report.*"
    
    return summary

if case:
    exec_summary = generate_executive_summary(case)
    print(exec_summary)

In [ ]:
# Export executive summary to file
if case:
    output_file = f"{case['case_id']}_executive_summary.md"
    with open(output_file, 'w') as f:
        f.write(exec_summary)
    print(f"Saved to: {output_file}")

## 7. Timeline Analysis

In [ ]:
def analyze_timeline(case: dict):
    """Analyze the engagement timeline."""
    timeline = case.get('timeline', [])
    if not timeline:
        print("No timeline data available.")
        return
    
    print(f"=== Timeline ({len(timeline)} events) ===")
    print()
    
    tools_used = Counter(e.get('tool') for e in timeline if e.get('tool'))
    print("Tools Used:")
    for tool, count in tools_used.most_common():
        print(f"  {tool}: {count}x")
    
    print("\nRecent Activity:")
    for entry in timeline[-10:]:
        ts = entry.get('timestamp', '')[:19].replace('T', ' ')
        action = entry.get('action', 'unknown')
        target = entry.get('target', '')
        print(f"  [{ts}] {action}" + (f" -> {target}" if target else ""))

if case:
    analyze_timeline(case)

## 8. Export Utilities

In [ ]:
def export_to_csv(case: dict, output_prefix: str = None):
    """Export case data to CSV files."""
    import csv
    
    prefix = output_prefix or case['case_id']
    
    # Export findings
    findings_file = f"{prefix}_findings.csv"
    with open(findings_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'severity', 'type', 'title', 'target', 'description', 'timestamp'])
        writer.writeheader()
        for finding in case.get('findings', []):
            writer.writerow({
                'id': finding.get('id'),
                'severity': finding.get('severity'),
                'type': finding.get('type'),
                'title': finding.get('title'),
                'target': finding.get('target'),
                'description': finding.get('description', '')[:500],
                'timestamp': finding.get('timestamp')
            })
    print(f"Findings exported to: {findings_file}")
    
    # Export ports
    ports_file = f"{prefix}_ports.csv"
    with open(ports_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['target', 'port', 'protocol', 'state', 'service', 'version'])
        writer.writeheader()
        for addr, data in case.get('targets', {}).items():
            for port in data.get('ports', []):
                writer.writerow({
                    'target': addr,
                    'port': port.get('port'),
                    'protocol': port.get('protocol'),
                    'state': port.get('state'),
                    'service': port.get('service'),
                    'version': port.get('version')
                })
    print(f"Ports exported to: {ports_file}")

if case:
    export_to_csv(case)

---

## Quick Reference

```python
# Load a case
case = load_case("abc12345")

# List all cases
cases = list_all_cases()

# Analyze targets
analysis = analyze_targets(case)

# Analyze findings
findings = analyze_findings(case)

# Cross-case analysis
cross = cross_case_analysis()

# Generate report
summary = generate_executive_summary(case)

# Export to CSV
export_to_csv(case)
```